In [1]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"]="7"
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import math
import time
import scanpy as sc
from tqdm.notebook import tqdm
import gc
import random
import copy
import pickle
import warnings
warnings.filterwarnings("ignore")

from gANCHOR.module3_relapse import RELAPSE_LABEL_MAP, PatientResponseModel, response_train
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, multilabel_confusion_matrix

from matplotlib import rcParams
rcParams['figure.figsize']=(10, 10)

In [2]:
PATH = '../results'

model_name = 'gANCHOR_2026-08-04_02:53:18'
fn = f'{PATH}/{model_name}'

In [3]:
m = sc.read_h5ad(f"{fn}/gANCHOR_moduleI_outputs.h5ad")
m

AnnData object with n_obs × n_vars = 1693727 × 11140
    obs: 'cell_type', 'assay', 'patient', 'sample_id', 'data_type', 'batch_correct', 'Response', 'sample_id_old', 'percent_ribo', 'n_counts', 'barcode'
    var: 'gene_median'
    uns: 'data_yuniques', 'log1p', 'patients_split', 'type_yuniques'
    obsm: 'cell_embed', 'gene_pred', 'type_prob'

In [4]:
patient_info = pd.read_table("table/patient_features.txt", sep='\t', index_col = 0, header = 0, encoding = "utf-8")
patient_info

,Data_id,Patient_id_in_paper,Patient_id,Response_in_paper,Response,Data_id_old,Cell_num,Relapse
0,CART_Deng 2020,ac01,0,CR,R,1,6686,not mentioned
1,CART_Deng 2020,ac02,1,PD,NR,1,8573,NR
2,CART_Deng 2020,ac03,2,PD,NR,1,4459,NR
3,CART_Deng 2020,ac04,3,PD,NR,1,2801,NR
4,CART_Deng 2020,ac05,4,CR,R,1,1732,not mentioned
...,...,...,...,...,...,...,...,...
158,CART_Bai 2024,DC78,DC78,CR,R,8,3997,No
159,CART_Bai 2024,DC79,DC79,NR,NR,8,3590,NR
160,CART_Bai 2024,DC80,DC80,RL+,R,8,5097,RL
161,CART_Bai 2024,DC81,DC81,RL+,R,8,3233,RL


In [5]:
train_patients, valid_patients, test_patients = m.uns['patients_split']['train'], m.uns['patients_split']['val'], m.uns['patients_split']['test']
len(train_patients), len(valid_patients), len(test_patients)

(101, 26, 34)

In [7]:
relapse_label_map = {
    "No": 0,   # no relapse
    "RL": 1,   # relapse
}

if relapse_label_map != RELAPSE_LABEL_MAP:
    raise RuntimeError("The notebook relapse_label_map does not match module3_relapse.py.")

relapse_categories = list(relapse_label_map)

## Standardize patient IDs and relapse labels.
patient_info = patient_info.copy()
patient_info["Patient_id"] = (patient_info["Patient_id"].astype(str).str.strip())
patient_info["Relapse"] = (patient_info["Relapse"].astype(str).str.strip())

relapse_info = patient_info.loc[
    patient_info["Relapse"].isin(relapse_categories),
    ["Patient_id_in_paper", "Patient_id", "Relapse", "Cell_num"],
].copy()


relapse_info = (relapse_info.drop_duplicates(subset="Patient_id", keep="first").reset_index(drop=True))
relapse_info["Relapse_index"] = (relapse_info["Relapse"].map(relapse_label_map).astype(np.int64))

patient_relapse_dict = (relapse_info.set_index("Patient_id")["Relapse_index"].to_dict())

def filter_relapse_patients(split_patients, relapse_table, id_col="Patient_id"):
    split_patients = np.asarray(split_patients).astype(str)
    split_patients = np.char.strip(split_patients)

    selected_table = relapse_table[relapse_table[id_col].isin(split_patients)].copy()

    patient_order = {patient_id: position for position, patient_id in enumerate(split_patients)}
    selected_table["_order"] = selected_table[id_col].map(patient_order)
    selected_table = selected_table.sort_values("_order").drop(columns="_order").reset_index(drop=True)

    selected_patients = selected_table[id_col].to_numpy()

    return selected_patients, selected_table

relapse_train_patients, relapse_train_info = filter_relapse_patients(train_patients, relapse_info)
relapse_valid_patients, relapse_valid_info = filter_relapse_patients(valid_patients, relapse_info)
relapse_test_patients, relapse_test_info = filter_relapse_patients(test_patients, relapse_info)

print("Relapse label map:", relapse_label_map)
print("Relapse train patients:", len(relapse_train_patients))
print("Relapse validation patients:", len(relapse_valid_patients))
print("Relapse test patients:", len(relapse_test_patients))

for split_name, split_info in (("Training", relapse_train_info), ("Validation", relapse_valid_info), ("Test", relapse_test_info),):
    distribution = (split_info["Relapse"].value_counts().reindex(relapse_categories, fill_value=0))
    print(f"\n{split_name} relapse distribution:")
    print(distribution)

Relapse label map: {'No': 0, 'RL': 1}
Relapse train patients: 72
Relapse validation patients: 17
Relapse test patients: 22

Training relapse distribution:
Relapse
No    35
RL    37
Name: count, dtype: int64

Validation relapse distribution:
Relapse
No     7
RL    10
Name: count, dtype: int64

Test relapse distribution:
Relapse
No     8
RL    14
Name: count, dtype: int64


In [8]:
max_cell_num = 2000

random.seed(0)
cell_pick = []
cell_unpick = []
obs_patient_ids = m.obs["patient"].astype(str).str.strip()

for patient in list(relapse_train_patients) + list(relapse_valid_patients) + list(relapse_test_patients):
        
    patient_id = str(patient).strip()
    cell_patients = m.obs.index[obs_patient_ids.eq(patient_id)].tolist()

    if len(cell_patients) <= max_cell_num:
        selected = cell_patients
        unselected = []
    else:
        selected = random.sample(cell_patients, max_cell_num)
        selected_set = set(selected)
        unselected = [cell_id for cell_id in cell_patients if cell_id not in selected_set]

    cell_pick.extend(selected)
    cell_unpick.extend(unselected)
    
split_cell_map = {idx: 'pick' for idx in cell_pick}
split_cell_map.update({idx: 'unpick' for idx in cell_unpick})
m.obs['relapse_pick'] = m.obs.index.map(split_cell_map)

print("Picked relapse cells:", len(cell_pick))
print("Unpicked relapse cells:", len(cell_unpick))
m

Picked relapse cells: 208345
Unpicked relapse cells: 311139


AnnData object with n_obs × n_vars = 1693727 × 11140
    obs: 'cell_type', 'assay', 'patient', 'sample_id', 'data_type', 'batch_correct', 'Response', 'sample_id_old', 'percent_ribo', 'n_counts', 'barcode', 'relapse_pick'
    var: 'gene_median'
    uns: 'data_yuniques', 'log1p', 'patients_split', 'type_yuniques'
    obsm: 'cell_embed', 'gene_pred', 'type_prob'

In [9]:
relapse_class_indices = list(relapse_label_map.values())
train_class_counts = (relapse_train_info["Relapse_index"].value_counts().reindex(relapse_class_indices, fill_value=0).sort_index())

if (train_class_counts == 0).any():
    missing_indices = train_class_counts[train_class_counts == 0].index.tolist()
    missing_labels = [label for label, index in relapse_label_map.items() if index in missing_indices]
    raise ValueError(f"Training split is missing relapse classes: {missing_labels}")

relapse_class_weights = (len(relapse_train_info) / (len(relapse_class_indices) * train_class_counts)).astype(float).tolist()

print("Relapse class weights:", {label: relapse_class_weights[index] for label, index in relapse_label_map.items()},)

Relapse class weights: {'No': 1.0285714285714285, 'RL': 0.972972972972973}


In [10]:
device = torch.device('cuda')

epochs = 1000
scheduler_patience = 50
factor = 0.1
eps = 1e-08
early_stopping = 120

seeds = [1, 2, 3, 4]
lrs = [0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.005, 0.01]
weight_decays = [0.01, 0.001, 0.0001]


def calculate_f1(true_indices, predicted_indices):
    return f1_score(
        true_indices.numpy(),
        predicted_indices.numpy(),
        labels=relapse_class_indices,
        average="binary",
        zero_division=0,
    )


saved_results = []

for seed in tqdm(seeds, desc="Runs", leave=False):
    for weight_decay in tqdm(weight_decays, desc="Weight decay", leave=False):
        for lr in tqdm(lrs, desc="Learning rate", leave=False):

            np.random.seed(seed)
            
            model = PatientResponseModel(m.obsm["cell_embed"].shape[1], max_cell=max_cell_num, device=device, fc_dropout=0.2, seed=seed).to(device)

            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay, eps=eps)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=factor, patience=scheduler_patience)
            
            (tr_pa_pred, 
             val_pa_pred, 
             test_pa_pred, 
             tr_pa_GT, 
             val_pa_GT, 
             test_pa_GT, 
             tr_pa_cell_wt, 
             val_pa_cell_wt, 
             test_pa_cell_wt, 
             epoch
            ) = response_train(model,
                optimizer,
                m,
                relapse_train_patients,
                relapse_valid_patients,
                relapse_test_patients,
                max_cell=max_cell_num,
                scheduler=scheduler,
                device=device,
                epochs=epochs,
                patience=early_stopping,
                response_weights=relapse_class_weights,
                patient_relapse_dict=patient_relapse_dict,
                cell_selection_column="relapse_pick",
            )

            tr_pred = tr_pa_pred.argmax(dim=1)
            val_pred = val_pa_pred.argmax(dim=1)
            test_pred = test_pa_pred.argmax(dim=1)

            f1_macro_tr = calculate_f1(tr_pa_GT, tr_pred)
            f1_macro_val = calculate_f1(val_pa_GT, val_pred)
            f1_macro_test = calculate_f1(test_pa_GT, test_pred)

            saved_results.append([seed, lr, weight_decay, tr_pa_pred.tolist(), val_pa_pred.tolist(), test_pa_pred.tolist(), tr_pa_GT.tolist(), val_pa_GT.tolist(), test_pa_GT.tolist(), tr_pa_cell_wt.tolist(),  
                                  val_pa_cell_wt.tolist(), test_pa_cell_wt.tolist(), f1_macro_tr, f1_macro_val, f1_macro_test])

Runs:   0%|          | 0/4 [00:00<?, ?it/s]

Weight decay:   0%|          | 0/3 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

...

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Weight decay:   0%|          | 0/3 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

Learning rate:   0%|          | 0/7 [00:00<?, ?it/s]

In [19]:
from datetime import date
from time import gmtime, strftime

saved_time = strftime("%Y-%m-%d_%H:%M:%S", gmtime())
fn1 = f"{fn}/relapse_{saved_time}"

if os.path.exists(fn1) is not True:
    os.mkdir(fn1)
    
saved_results_table = pd.DataFrame([row for row in saved_results], columns=['seed', 'lr', 'we_de', 'tr_pred', 'val_pred', 'test_pred', 'tr_GT', 'val_GT', 
                                                                            'test_GT', 'tr_cell_wt', 'val_cell_wt', 'test_cell_wt', 'f1_tr', 'f1_val', 'f1_test'])

file = open(f'{fn1}/gANCHOR_moduleIII_relapse_outputs.pkl','wb')
pickle.dump(saved_results_table, file, protocol=4)
file.close()

In [20]:
m.obs[['barcode', 'relapse_pick']].to_csv(f'{fn1}/relapse_cell_pick.csv', index=True, header=True)